# 00 — Preparar a execução de Iatan 2

Confere o pacote de insumos e prepara a raiz usada pelos notebooks 01–05. Os dados preservados permitem começar pelo 02. Use o 01 para reexecutar a aquisição com a chave da EPA.

In [ ]:
# Registros do experimento usam caminhos relativos à raiz configurada.
# Versões de bibliotecas e resultados são preservados sem identidade da sessão.
import builtins as _builtins
import json as _json
import os as _os
import re as _re
import warnings as _warnings
from pathlib import Path as _Path

def _project_root():
    for name in ("PROJECT_ROOT", "ROOT33", "ROOT", "PROJECT", "SOURCE", "root"):
        value = globals().get(name)
        if isinstance(value, _Path) and value.is_dir():
            return value.resolve()
    return _Path.cwd()

def portable_path(path, root=None):
    path = _Path(path).expanduser()
    root = _Path(root) if root is not None else _project_root()
    try:
        return path.resolve().relative_to(root.resolve()).as_posix()
    except (ValueError, OSError):
        return "external/" + path.name

def public_record(value):
    if isinstance(value, _Path):
        return portable_path(value)
    if isinstance(value, dict):
        return {k: public_record(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [public_record(v) for v in value]
    if isinstance(value, str):
        root = _project_root()
        for prefix in {str(root), root.as_posix()}:
            if value.startswith(prefix + "/") or value.startswith(prefix + "\\"):
                return value[len(prefix)+1:].replace("\\", "/")
            value = value.replace(prefix + "/", "").replace(prefix + "\\", "")
            value = value.replace(prefix, ".")
        if _re.match(r"^(?:[A-Za-z]:[\\/]|/(?:content|tmp|home|Users|mnt)/)", value):
            return "external/" + value.replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]
        return value
    # Cópias de tabelas exportadas normalizam somente colunas textuais.
    if type(value).__module__.startswith("pandas.") and hasattr(value, "columns"):
        result = value.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = result[column].map(public_record)
        return result
    return value

def _project_print(*values, **kwargs):
    root = _project_root()
    if root.is_dir():
        _os.chdir(root)
    return _builtins.print(*(public_record(v) for v in values), **kwargs)

def _project_dump(value, *args, **kwargs):
    return _json.dump(public_record(value), *args, **kwargs)

def _project_dumps(value, *args, **kwargs):
    return _json.dumps(public_record(value), *args, **kwargs)

def _project_csv(frame, *args, **kwargs):
    return public_record(frame).to_csv(*args, **kwargs)

def _project_write_text(path, text, *args, **kwargs):
    return path.write_text(public_record(text), *args, **kwargs)

def _project_display(*values, **kwargs):
    from IPython.display import display as _display
    return _display(*(public_record(v) for v in values), **kwargs)

# Avisos mantêm o conteúdo sem diretórios temporários ou IDs do kernel.
_warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {public_record(str(message))}\n"
)
_os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
_context = _json.loads(_os.environ.get("MLFLOW_RUN_CONTEXT", "{}"))
_context.update({
    "mlflow.user": "equipe", "mlflow.source.name": "iatan",
    "mlflow.source.type": "LOCAL", "mlflow.source.git.commit": "",
    "mlflow.source.git.branch": "", "mlflow.source.git.repoURL": "",
    "mlflow.source.git.diff": "", "mlflow.source.git.dirty": "",
})
_os.environ["MLFLOW_RUN_CONTEXT"] = _json.dumps(_context)


from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import os
import shutil

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")

default_home = '/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos/Iatan 2025 - entrega revisada' if drive is not None else str(Path.cwd())
delivery = Path(os.environ.get("IATAN_DELIVERY_ROOT", default_home)).expanduser().resolve()
root = Path(os.environ.get("IATAN_PROJECT_ROOT", delivery / "execucao")).expanduser().resolve()
source_zip = delivery / "Iatan2025_insumos.zip"
expected_sha = '32499f1af9789d9df2df4dd1748ded328232cfae6f59a6ac4e8ca769bfd12594'
def sha256(data):
    return hashlib.sha256(data).hexdigest()

if sha256(source_zip.read_bytes()) != expected_sha:
    raise ValueError("O arquivo de insumos diverge do pacote da entrega.")

with ZipFile(source_zip) as package:
    manifest = json.loads(package.read("manifesto_insumos.json"))
    entries = []
    for item in manifest["files"]:
        destination = (root / item["path"]).resolve()
        data = package.read(item["path"])
        if not destination.is_relative_to(root) or sha256(data) != item["sha256"]:
            raise ValueError(f"Entrada inválida: {item['path']}")
        if destination.exists() and sha256(destination.read_bytes()) != item["sha256"]:
            raise FileExistsError(f"Arquivo de uma execução existente: {destination}. Use outra raiz IATAN_PROJECT_ROOT.")
        entries.append((destination, data))

notebooks = sorted((delivery / "Notebooks").glob("0[1-5]_*.ipynb"))
if len(notebooks) != 5:
    raise ValueError("A pasta Notebooks deve conter os cinco notebooks revisados.")
for notebook in notebooks:
    target = root / "notebooks" / notebook.name
    if target.exists() and target.read_bytes() != notebook.read_bytes():
        raise FileExistsError(f"Notebook existente difere da entrega: {target}. Use outra raiz.")

for destination, data in entries:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        destination.write_bytes(data)
for notebook in notebooks:
    target = root / "notebooks" / notebook.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copy2(notebook, target)
_project_write_text(root / "manifesto_insumos.json", _project_dumps(manifest, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
os.environ["IATAN_PROJECT_ROOT"] = str(root)
_project_print("Insumos conferidos. Raiz da execução:", root)

## Ambiente

Instale as dependências no ambiente que executará os notebooks. Em outra sessão do Colab, execute a célula de instalação novamente. Se o Colab pedir uma reinicialização após instalar dependências, reinicie a sessão antes de importar os pacotes.

In [ ]:
import subprocess
import sys

requirements = delivery / "requirements-iatan2025.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements), "psutil==7.2.2"])